In [10]:
import pandas as pd
import numpy as np
import duckdb

In [2]:
# Cell 2: load calender, sales_evaluation, and prices dataset
calendar_df = pd.read_csv("../data/raw/calendar.csv")
prices_df = pd.read_csv("../data/raw/sell_prices.csv")
sales_eval_df = pd.read_csv("../data/raw/sales_train_evaluation.csv")

print("5 Rows of Calender df:\n", calendar_df.head())
print("\n5 Rows Prices df:\n", prices_df.head())
print("\n5 Rows Sales Evaluation:\n", sales_eval_df.head())


5 Rows of Calender df:
          date  wm_yr_wk    weekday  wday  month  year    d event_name_1  \
0  2011-01-29     11101   Saturday     1      1  2011  d_1          NaN   
1  2011-01-30     11101     Sunday     2      1  2011  d_2          NaN   
2  2011-01-31     11101     Monday     3      1  2011  d_3          NaN   
3  2011-02-01     11101    Tuesday     4      2  2011  d_4          NaN   
4  2011-02-02     11101  Wednesday     5      2  2011  d_5          NaN   

  event_type_1 event_name_2 event_type_2  snap_CA  snap_TX  snap_WI  
0          NaN          NaN          NaN        0        0        0  
1          NaN          NaN          NaN        0        0        0  
2          NaN          NaN          NaN        0        0        0  
3          NaN          NaN          NaN        1        1        0  
4          NaN          NaN          NaN        1        0        1  

5 Rows Prices df:
   store_id        item_id  wm_yr_wk  sell_price
0     CA_1  HOBBIES_1_001     11325  

### **Filter to one store (CA_1)**

In [3]:
# Cell 3: Filter to one store (CA_1)

STORE = "CA_1"

id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [c for c in sales_eval_df.columns if c.startswith("d_")]

sales_s = sales_eval_df[sales_eval_df["store_id"] == STORE].reset_index(drop=True)
prices_s = prices_df[prices_df["store_id"] == STORE].reset_index(drop=True)

print("day columns  :", len(day_cols), "| first/last:", day_cols[0], day_cols[-1])
print("sales_s shape :", sales_s.shape)
print("prices_s shape:", prices_s.shape)

del sales_eval_df

day columns  : 1941 | first/last: d_1 d_1941
sales_s shape : (3049, 1947)
prices_s shape: (698412, 4)


## Why we use only one store (CA_1)

**The problem**
- M5 has 10 stores x 3,049 items = 30,490 time series, each with 1,941 days.
- After reshaping from wide to long, that is about **59 million rows**.
- On a laptop this can crash the notebook or make every step very slow.

**The decision**
- We use **one store: CA_1** (California store 1).
- That gives 3,049 items x 1,941 days = about **5.9 million rows**, which fits in memory.

**Why this is still a valid project**
- The task is the same: forecast daily sales per item for the next 28 days.
- CA_1 still has all 3 categories (HOBBIES, HOUSEHOLD, FOODS), all 7 departments, real prices, events and SNAP days.
- Faster experiments: backtesting, tuning and SHAP all need many reruns, and 10x less data makes each run quick.
- Hyperparameters, features and the pipeline built on one store carry over to the other stores.

**Why CA_1 specifically**
- It is the first store in the data, so it is easy to reproduce.
- California has its own SNAP schedule, so we use the `snap_CA` flag.

**Limitations (to state in the README)**
- Results are for one store only. Another store may behave differently.
- We cannot study store-level or state-level hierarchy (item vs store vs state) with one store.
- Cross-store effects are not learned, for example item popularity across stores.

**Future work**
- Make `STORE` a parameter in `params.yaml` and rerun the pipeline for other stores.
- Scale to all 10 stores with chunked processing or a larger machine.

### **Wide to long(melt)**

In [4]:
# cell 4: Shrink memory first: categories for ids, int16 for sales

for c in id_cols:
    sales_s[c] = sales_s[c].astype("category")
sales_s[day_cols] = sales_s[day_cols].astype("int16")

# wide -> long
df = sales_s.melt(id_vars= id_cols, value_vars=day_cols,
                  value_name="sales", var_name="d")


# d_1 -> 1 (cheap integer day index instead of a string)
df["d_num"] = df["d"].astype(str).str.replace("d_", "", regex=False)
df["d_num"] = pd.to_numeric(df["d_num"], errors="raise").astype("int16")
df = df.drop(columns="d")

print("shape:", df.shape)
print("days :", df["d_num"].min(), "to", df["d_num"].max())
print("items:", df["item_id"].nunique())
display(df.head())

shape: (5918109, 8)
days : 1 to 1941
items: 3049


,id,item_id,dept_id,cat_id,store_id,state_id,sales,d_num
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,1
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,1
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,1


## Merge Calendar

- Right now each row only knows d_num (day 1, 2, 3...). It doesn’t know the real date, the weekday, the month, or whether that day was a holiday or a SNAP day.
- That information lives in calendar, so we join it on the day number.
calendar has its own d column (d_1, d_2...), so we make a d_num in the same way and join on that.
- calendar has SNAP flags for all 3 states (snap_CA, snap_TX, snap_WI). CA_1 is in California, so we keep only snap_CA and rename it to snap.
- A calendar has exactly one row per day, so the merge must not change the row count. We add an assert to prove it. If the row count grows, something is duplicated.

In [5]:
# Cell 5: Merge the calendar

STATE = "CA"   # CA_1 is a California store

cal = calendar_df.copy()
cal["d_num"] = cal["d"].astype(str).str.replace("d_", "", regex=False)
cal["d_num"] = pd.to_numeric(cal["d_num"], errors="raise").astype("int16")

cal = cal[["d_num", "date", "wm_yr_wk", "wday", "month", "year",
           "event_name_1", "event_type_1", "event_name_2", "event_type_2",
           f"snap_{STATE}"]].rename(columns={f"snap_{STATE}": "snap"})
n_before = len(df)
df = df.merge(cal, on="d_num", how="left")
assert len(df) == n_before, "merge changed the row count"

print("shape:", df.shape)
# convert the date column from string to real datetime
df["date"] = pd.to_datetime(df["date"])
print("date range:", df["date"].min().date(), "to", df["date"].max().date())
print("missing dates:", df["date"].isna().sum())
display(df.sample(5, random_state=42))

shape: (5918109, 18)
date range: 2011-01-29 to 2016-05-22
missing dates: 0


,id,item_id,dept_id,cat_id,store_id,state_id,sales,d_num,date,wm_yr_wk,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap
327374,HOUSEHOLD_2_035_CA_1_evaluation,HOUSEHOLD_2_035,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,0,108,2011-05-16,11116,3,5,2011,NaN,NaN,NaN,NaN,0
812768,FOODS_1_125_CA_1_evaluation,FOODS_1_125,FOODS_1,FOODS,CA_1,CA,0,267,2011-10-22,11139,1,10,2011,NaN,NaN,NaN,NaN,0
5539411,FOODS_3_203_CA_1_evaluation,FOODS_3_203,FOODS_3,FOODS,CA_1,CA,3,1817,2016-01-19,11551,4,1,2016,NaN,NaN,NaN,NaN,0
5136620,FOODS_2_278_CA_1_evaluation,FOODS_2_278,FOODS_2,FOODS,CA_1,CA,0,1685,2015-09-09,11532,5,9,2015,NaN,NaN,NaN,NaN,1
3179353,FOODS_3_071_CA_1_evaluation,FOODS_3_071,FOODS_3,FOODS,CA_1,CA,1,1043,2013-12-06,11345,7,12,2013,NaN,NaN,NaN,NaN,1


## Merge the prices

- Prices are weekly. sell_prices has one price per item per week (wm_yr_wk), not per day. So we join on item and week: item_id + wm_yr_wk. Every day in the same week gets the same price.
- prices_s is already CA_1 only, so we don’t need store_id in the join.
- The join must not change the row count, so we keep the assert.
Missing prices are expected. An item only has a price once it is on sale. Before that, sell_price is NaN. Those early zeros mean “not available yet”, not “no demand”. We handle this in EDA and in feature building.
- We convert prices_s["item_id"] to the same category type as df, so the merge stays fast and memory-light.

In [6]:
# cell 7: if prices were already merged, drop them first
df = df.drop(columns = [c for c in df.columns if c.startswith("sell_price")])

p = prices_s[["item_id", "wm_yr_wk", "sell_price"].copy()]
p["item_id"] = pd.Categorical(p["item_id"], categories=df["item_id"].cat.categories)

n_before = len(df)
df = df.merge(p, on = ["item_id", "wm_yr_wk"], how="left")
assert len(df) == n_before, "merge changed the row count"

print("shape:", df.shape)
print("Share of rows with missing price:", round(df["sell_price"].isna().mean(), 3))
print("items with at least one missing price:", df.loc[df["sell_price"].isna(), "item_id"].nunique())
display(df[["item_id", "d_num", "date", "sales", "sell_price"]].sample(5, random_state=1))

C:\Users\ASUS\AppData\Local\Temp\ipykernel_25828\2898650906.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  p["item_id"] = pd.Categorical(p["item_id"], categories=df["item_id"].cat.categories)


shape: (5918109, 19)
Share of rows with missing price: 0.191
items with at least one missing price: 1845


,item_id,d_num,date,sales,sell_price
3969170,FOODS_3_197,1302,2014-08-22,2,1.98
4879783,HOUSEHOLD_2_288,1601,2015-06-17,0,3.97
1998748,FOODS_1_043,656,2012-11-14,0,1.78
698245,HOBBIES_1_025,230,2011-09-15,0,3.74
4354003,HOBBIES_1_032,1429,2014-12-27,0,0.58


### **Fix dtypes, sort and save**

- Right now several columns use more memory than they need. We shrink them (int8 for small numbers, float32 for price, category for text) so the notebooks run fast.
- We sort by item_id and then d_num. Time-series features such as lags and rolling means assume each item’s days are in order, so this is required.
- We check that days are consecutive for every item, with no gaps.
- We save to parquet, which is smaller and much faster to load than CSV. This file is the output of Notebook 01, and every later notebook reads it.
- We reload the file and compare shapes to prove it saved correctly.- 

In [ ]:
assert df["sell_price"].isna().sum() > 1_000_000, "missing prices were filled"
assert (df["sell_price"].dropna() > 0).all(), "zero or negative prices found"

In [7]:
# Srink dtypes
df["wm_yr_wk"] = df["wm_yr_wk"].astype("int16")
df["wday"] = df["wday"].astype("int8")
df["month"] = df["month"].astype("int8")
df["year"] = df["year"].astype("int16")
df["snap"] = df["snap"].astype("int8")
df["sell_price"] = df["snap"].astype("float32")
for c in ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]:
    df[c] = df[c].astype("category")

# sort: each items's days in order
df = df.sort_values(["item_id", "d_num"]).reset_index(drop = True)

# check: every item has 9141 consecutive days
days_per_item = df.groupby("item_id", observed=True)["d_num"].agg(["count", "min", "max"])
assert (days_per_item["count"] == 1941).all(), "some items have missing days"
assert (days_per_item["min"] == 1).all() and (days_per_item["max"] == 1941).all()
print("every item has days 1 to 1941: OK")

print(df.dtypes)
print("memory (MB):", round(df.memory_usage(deep=True).sum() / 1e6, 1))

# save and verify
out_path = "../data/processed/ca1_long.parquet"
df.to_parquet(out_path, index=False)
check = pd.read_parquet(out_path)
print("saved:", out_path)
print("file size (MB):", round(out_path.stat().st_size / 1e6, 1))
print("reloaded shape:", check.shape, "| same as df:", check.shape == df.shape)

every item has days 1 to 1941: OK
id                    category
item_id               category
dept_id               category
cat_id                category
store_id              category
state_id              category
sales                    int16
d_num                    int16
date            datetime64[ns]
wm_yr_wk                 int16
wday                      int8
month                     int8
year                     int16
event_name_1          category
event_type_1          category
event_name_2          category
event_type_2          category
snap                      int8
sell_price             float32
dtype: object
memory (MB): 207.7
saved: ../data/processed/ca1_long.parquet


AttributeError: 'str' object has no attribute 'stat'

In [8]:
from pathlib import Path

out_path = Path("../data/processed/ca1_long.parquet")
print("absolute path :", out_path.resolve())
print("file size (MB):", round(out_path.stat().st_size / 1e6, 1))

check = pd.read_parquet(out_path)
print("reloaded shape:", check.shape, "| same as df:", check.shape == df.shape)
print("dtypes preserved:", (check.dtypes == df.dtypes).all())
del check

absolute path : C:\Users\ASUS\Desktop\AI_Projects\Retail-demand-forecasting\data\processed\ca1_long.parquet
file size (MB): 23.1
reloaded shape: (5918109, 19) | same as df: True
dtypes preserved: True


### **Use DuckDB (SQL) on the parquet file**

- DuckDB runs SQL directly on the parquet file, with no server and no database setup.
- It shows you can use SQL inside a forecasting workflow. It is also quick for summaries on 5.9 million rows.
- Query 1 is a quick overview of the data: total rows, items and days, total units sold, and the share of zero-sales days. Zeros dominate this kind of data (intermittent demand), which is why we need WAPE and MASE instead of MAPE.
- The overview shows the zero share twice: for all rows, and only for rows where the item had a price (so it was actually on sale). The second number is the fairer one.

In [ ]:
# Cell 9
PARQUET = out_path.resolve().as_posix()            # forward slashes work on Windows too
q1 = F"""
SELECT
    COUNT(*)                                       AS n_rows,
    COUNT(DISTINCT item_id)                        AS n_items,
    COUNT(DISTINCT d_num)                          AS n_days,
    SUM(sales)                                     AS total_units,
    ROUND(AVG(CAST(sales = 0 AS INTEGER)), 3)      AS zero_share_all_rows,
    ROUND(AVG(CASE WHEN sell_price IS NOT NULL
                   THEN CAST(sales = 0 AS INTEGER) END), 3) AS zero_share_when_on_sale
FROM read_parquet('{PARQUET}')
"""

display(duckdb.sql(q1).df())

,n_rows,n_items,n_days,total_units,zero_share_all_rows,zero_share_when_on_sale
0,5918109,3049,1941,7832248.0,0.638,0.638


In [14]:
# Cell 10
q2 = f"""
SELECT
    cat_id,
    COUNT(DISTINCT item_id)         AS n_items,
    SUM(sales)                      AS total_units,
    ROUND(AVG(sales), 3)            AS avg_units_per_item_day,
    ROUND(AVG((sales = 0)::INT), 3) AS zero_share
FROM read_parquet('{PARQUET}')
GROUP BY cat_id
ORDER BY total_units DESC
"""

display(duckdb.sql(q2).df())

,cat_id,n_items,total_units,avg_units_per_item_day,zero_share
0,FOODS,1437,5471661.0,1.962,0.569
1,HOUSEHOLD,1047,1468504.0,0.723,0.684
2,HOBBIES,565,892083.0,0.813,0.726


In [ ]:
# Cell 10
q3 = f"""
SELECT
    item_id,
    SUM(sales)           AS total_units,
    ROUND(AVG(sales), 2) AS avg_units_per_day
FROM read_parquet('{PARQUET}')
GROUP BY item_id
ORDER BY total_units DESC
LIMIT 10
"""
display(duckdb.sql(q3).df())

,item_id,total_units,avg_units_per_day
0,FOODS_3_090,128855.0,66.39
1,FOODS_3_586,88846.0,45.77
2,FOODS_3_252,76055.0,39.18
3,FOODS_3_120,63107.0,32.51
4,FOODS_3_714,54617.0,28.14
5,FOODS_3_587,52356.0,26.97
6,FOODS_3_808,46094.0,23.75
7,FOODS_2_197,41312.0,21.28
8,FOODS_3_080,39673.0,20.44
9,FOODS_3_555,38446.0,19.81


In [17]:
# Cell 11 (diagnostic):
q4 = f"""
SELECT
    COUNT(*) FILTER (WHERE sell_price IS NULL)               AS null_price_rows,
    COUNT(*) FILTER (WHERE isnan(sell_price))                AS nan_price_rows,
    SUM(sales) FILTER (WHERE sell_price IS NULL
                          OR isnan(sell_price))              AS units_when_no_price
FROM read_parquet('{PARQUET}')
"""
display(duckdb.sql(q4).df())

,null_price_rows,nan_price_rows,units_when_no_price
0,0,0,NaN


In [18]:
# Cell 11b (diagnostic):

chk = pd.read_parquet(out_path)

print("in memory  - missing prices:", df["sell_price"].isna().sum())
print("on disk    - missing prices:", chk["sell_price"].isna().sum())
print("in memory  - price == 0     :", (df["sell_price"] == 0).sum())
print("on disk    - price == 0     :", (chk["sell_price"] == 0).sum())
print("on disk    - min price      :", chk["sell_price"].min())
print("values identical:", df["sell_price"].equals(chk["sell_price"]))
del chk

in memory  - missing prices: 0
on disk    - missing prices: 0
in memory  - price == 0     : 3966749
on disk    - price == 0     : 3966749
on disk    - min price      : 0.0
values identical: True


In [19]:
# Cell 11c (diagnostic, changes nothing):

fresh = df[["item_id", "wm_yr_wk"]].merge(
    p, on=["item_id", "wm_yr_wk"], how="left")["sell_price"].astype("float32")

print("fresh  - missing prices   :", fresh.isna().sum())
print("fresh  - price == 0       :", (fresh == 0).sum())
print("current - price == 0      :", (df["sell_price"] == 0).sum())
print("rows where fresh has a real price but current is 0:",
      ((fresh.notna()) & (fresh != 0) & (df["sell_price"] == 0)).sum())
print("rows where current equals fresh (missing counted as 0):",
      round((fresh.fillna(0).values == df["sell_price"].values).mean(), 4))

fresh  - missing prices   : 1129842
fresh  - price == 0       : 0
current - price == 0      : 3966749
rows where fresh has a real price but current is 0: 3213165
rows where current equals fresh (missing counted as 0): 0.1337


In [20]:
df["sell_price"] = fresh.values
print("missing prices restored:", df["sell_price"].isna().sum())
df.to_parquet(out_path, index=False)

missing prices restored: 1129842


In [21]:
# Cell 11d (verification):
chk = pd.read_parquet(out_path)

# 1) price: missing is back, no zeros
print("price missing :", chk["sell_price"].isna().sum())        # expect 1129842
print("price == 0    :", (chk["sell_price"] == 0).sum())        # expect 0
print("min real price:", chk["sell_price"].min())

# 2) sales still equals the raw sales table
raw_total = int(sales_s[day_cols].sum().sum())
print("sales total raw vs saved:", raw_total, chk["sales"].sum(), raw_total == chk["sales"].sum())

# 3) shape
print("shape:", chk.shape)                                      # expect (5918109, 19)
del chk

price missing : 1129842
price == 0    : 0
min real price: 0.01
sales total raw vs saved: 7832248 7832248 True
shape: (5918109, 19)


In [22]:
display(duckdb.sql(q1).df())

,n_rows,n_items,n_days,total_units,zero_share_all_rows,zero_share_when_on_sale
0,5918109,3049,1941,7832248.0,0.638,0.552


## Summary of Notebook 01

**What we did**
- Loaded `sales_train_evaluation.csv`, `calendar.csv` and `sell_prices.csv`
- Kept one store (CA_1): 3,049 items x 1,941 days = 5,918,109 rows
- Reshaped wide to long, merged calendar (events, SNAP) and weekly prices
- Shrunk dtypes (207.7 MB in memory) and saved `data/processed/ca1_long.parquet` (23.1 MB)

**Checks passed**
- Every item has days 1 to 1941, with no gaps
- Total units match the raw file exactly (7,832,248)
- Prices: 1,129,842 rows (19.1%) have no price, kept as NaN, with no zero prices

**Key findings**
- Date range of the sales data: 2011-01-29 to 2016-05-22
- 63.8% of item-days have zero sales (55.2% when the item was on sale), so demand is intermittent and we use WAPE and MASE, not MAPE
- FOODS is about 70% of units and has the lowest zero share; HOBBIES has the highest (72.6%)
- The top 10 items are all FOODS and make up about 8% of units, so they will dominate WAPE

**Lessons learned**
- Missing price means "not on sale yet", not price 0
- A reload check must compare values, not only shape and dtypes

**Next:** Notebook 02, EDA